# Comparaison TF-IDF + Logistic Regression vs DistilBERT (fine-tuné)

Ce notebook complète le projet **Sentiment Analysis Pro** : le dashboard/API utilise un pipeline classique
(TF-IDF + Logistic Regression, voir `models/train_models.py`), performant et rapide à entraîner/servir.

Objectif ici : voir ce qu'apporte (ou pas) un modèle Transformer moderne (`distilbert-base-uncased`) fine-tuné
sur les mêmes données, et comparer accuracy / F1 / temps d'entraînement / temps d'inférence.

**Important** : pour que ce notebook tourne en quelques minutes sur CPU ou un petit GPU, on entraîne sur un
**sous-échantillon** du dataset IMDB (par défaut 4 000 avis train / 1 000 test). Pour un run complet sur les
50 000 avis, augmente `TRAIN_SAMPLE_SIZE` et `TEST_SAMPLE_SIZE` (prévoir un GPU et plusieurs dizaines de minutes).

In [2]:
%pip install -q transformers datasets torch scikit-learn evaluate accelerate

ERROR: Could not install packages due to an OSError: [WinError 32] The process cannot access the file because it is being used by another process: 'c:\\Users\\HP\\Downloads\\sentiment-analysis-pro\\.venv\\Lib\\site-packages\\transformers\\models\\xlm_roberta_xl\\modeling_xlm_roberta_xl.py'
Check the permissions.


[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import time
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.model_selection import train_test_split
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)

# Sous-échantillon pour un entraînement rapide (voir cellule markdown ci-dessus)
TRAIN_SAMPLE_SIZE = 4000
TEST_SAMPLE_SIZE = 1000
RANDOM_STATE = 42

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device utilisé :", device)

c:\Users\HP\Downloads\sentiment-analysis-pro\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device utilisé : cpu


## 1. Chargement des données (même source que `train_models.py`)

In [4]:
dataset = load_dataset("imdb")

# Mêmes 50 000 avis labellisés que dans models/train_models.py (train + test HF combinés)
full_df = pd.concat(
    [pd.DataFrame(dataset["train"]), pd.DataFrame(dataset["test"])],
    ignore_index=True,
)
full_df["sentiment"] = full_df["label"]  # 0 = negative, 1 = positive

# On garde le MEME split 80/20 que le pipeline classique pour une comparaison honnête
train_full, test_full = train_test_split(
    full_df, test_size=0.2, random_state=RANDOM_STATE, stratify=full_df["sentiment"]
)

# Puis on sous-échantillonne pour le fine-tuning DistilBERT (rapidité)
train_df = train_full.sample(n=TRAIN_SAMPLE_SIZE, random_state=RANDOM_STATE).reset_index(drop=True)
test_df = test_full.sample(n=TEST_SAMPLE_SIZE, random_state=RANDOM_STATE).reset_index(drop=True)

print(f"Train (sous-échantillon) : {len(train_df)} avis")
print(f"Test (sous-échantillon)  : {len(test_df)} avis")
train_df.head()

Using the latest cached version of the dataset since imdb couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'plain_text' at C:\Users\HP\.cache\huggingface\datasets\imdb\plain_text\0.0.0\e6281661ce1c48d982bc483cf8a173c1bbeb5d31 (last modified on Wed Apr  1 21:45:50 2026).


Train (sous-échantillon) : 4000 avis
Test (sous-échantillon)  : 1000 avis


,text,label,sentiment
0,I've seen comments from Turkish people (which ...,0,0
1,"Andrewjlau, I could not agree more. My girlfri...",0,0
2,I have a lot of time for all the Columbo films...,1,1
3,This is about some vampires (who can run aroun...,0,0
4,Steve Martin should quit trying to do remakes ...,0,0


## 2. Baseline — TF-IDF + Logistic Regression (même sous-échantillon, pour une comparaison équitable)

In [5]:
tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))
X_train_tfidf = tfidf.fit_transform(train_df["text"])
X_test_tfidf = tfidf.transform(test_df["text"])

start = time.time()
logreg = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
logreg.fit(X_train_tfidf, train_df["sentiment"])
tfidf_train_time = time.time() - start

start = time.time()
tfidf_preds = logreg.predict(X_test_tfidf)
tfidf_inference_time = (time.time() - start) / len(test_df)  # secondes / avis

tfidf_acc = accuracy_score(test_df["sentiment"], tfidf_preds)
tfidf_p, tfidf_r, tfidf_f1, _ = precision_recall_fscore_support(
    test_df["sentiment"], tfidf_preds, average="binary"
)

print(f"Accuracy  : {tfidf_acc:.4f}")
print(f"F1-score  : {tfidf_f1:.4f}")
print(f"Temps entraînement : {tfidf_train_time:.2f}s")
print(f"Temps inférence    : {tfidf_inference_time*1000:.3f} ms/avis")

Accuracy  : 0.8520
F1-score  : 0.8563
Temps entraînement : 0.16s
Temps inférence    : 0.001 ms/avis


## 3. Fine-tuning DistilBERT

`distilbert-base-uncased` : version distillée de BERT (~60% plus légère, ~97% des performances), un bon compromis
pour un premier essai Transformer.

In [6]:
MODEL_NAME = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=256)

from datasets import Dataset

train_ds = Dataset.from_pandas(train_df[["text", "sentiment"]].rename(columns={"sentiment": "label"}))
test_ds = Dataset.from_pandas(test_df[["text", "sentiment"]].rename(columns={"sentiment": "label"}))

train_ds = train_ds.map(tokenize, batched=True)
test_ds = test_ds.map(tokenize, batched=True)

train_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
test_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

c:\Users\HP\Downloads\sentiment-analysis-pro\.venv\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\HP\.cache\huggingface\hub\models--distilbert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Map: 100%|██████████| 1000/1000 [00:00<00:00, 1524.50 examples/s]


In [7]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(device)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
    return {"accuracy": acc, "precision": p, "recall": r, "f1": f1}

training_args = TrainingArguments(
    output_dir="./distilbert_sentiment_checkpoints",
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=50,
    learning_rate=2e-5,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    compute_metrics=compute_metrics,
)

start = time.time()
trainer.train()
bert_train_time = time.time() - start
print(f"Temps d'entraînement DistilBERT : {bert_train_time:.2f}s")

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 3043.87it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
c:\Users\HP\Downloads\sentiment-analysis-pro\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as 

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.310146,0.297974,0.880000,0.899796,0.861057,0.880000
2,0.178953,0.333369,0.888000,0.906314,0.870841,0.888224


c:\Users\HP\Downloads\sentiment-analysis-pro\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Temps d'entraînement DistilBERT : 9790.75s


In [8]:
start = time.time()
bert_results = trainer.evaluate()
bert_inference_time = (time.time() - start) / len(test_df)  # secondes / avis

print(f"Accuracy  : {bert_results['eval_accuracy']:.4f}")
print(f"F1-score  : {bert_results['eval_f1']:.4f}")
print(f"Temps entraînement : {bert_train_time:.2f}s")
print(f"Temps inférence    : {bert_inference_time*1000:.3f} ms/avis")

c:\Users\HP\Downloads\sentiment-analysis-pro\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.178953,0.333369,2,0.888000,0.906314,0.870841,0.888224


Accuracy  : 0.8880
F1-score  : 0.8882
Temps entraînement : 9790.75s
Temps inférence    : 204.111 ms/avis


## 4. Tableau comparatif final

In [9]:
comparison = pd.DataFrame([
    {
        "Modèle": "TF-IDF + Logistic Regression",
        "Accuracy": round(tfidf_acc, 4),
        "F1-score": round(tfidf_f1, 4),
        "Temps entraînement (s)": round(tfidf_train_time, 2),
        "Temps inférence (ms/avis)": round(tfidf_inference_time * 1000, 3),
    },
    {
        "Modèle": "DistilBERT (fine-tuné)",
        "Accuracy": round(bert_results["eval_accuracy"], 4),
        "F1-score": round(bert_results["eval_f1"], 4),
        "Temps entraînement (s)": round(bert_train_time, 2),
        "Temps inférence (ms/avis)": round(bert_inference_time * 1000, 3),
    },
])
comparison

,Modèle,Accuracy,F1-score,Temps entraînement (s),Temps inférence (ms/avis)
0,TF-IDF + Logistic Regression,0.852,0.8563,0.16,0.001
1,DistilBERT (fine-tuné),0.888,0.8882,9790.75,204.111


## 5. Conclusion

À compléter avec tes propres résultats une fois le notebook exécuté, par exemple :

- **Écart de performance** : DistilBERT gagne généralement quelques points d'accuracy/F1 grâce à la compréhension
  contextuelle (négations, ironie, structures de phrase complexes) que TF-IDF ne capture pas.
- **Coût** : DistilBERT est nettement plus lent à entraîner et à servir (CPU notamment), et nécessite plus de
  mémoire — à mettre en balance avec le gain de performance selon le cas d'usage.
- **Choix de production** : pour ce projet, TF-IDF + Logistic Regression reste un choix pertinent (rapide, léger,
  interprétable avec LIME, suffisant en performance) — DistilBERT serait plus justifié si l'accuracy devenait
  critique et qu'un GPU est disponible en production.

*(Remplace ce paragraphe par tes vrais chiffres et ton analyse après exécution.)*

## 6. Publier ce résultat dans le dashboard de l'app

La cellule ci-dessous écrit les résultats dans `saved_models/transformer_reference.json`. Redémarre `python app.py` (ou rafraîchis la page) : l'onglet **Modèles** du dashboard affichera automatiquement une ligne "DistilBERT (fine-tuné)" marquée **RÉFÉRENCE\***, sans jamais entrer en compétition avec le modèle réellement déployé.

In [10]:
# Écrit les résultats dans saved_models/transformer_reference.json pour que
# le dashboard de l'app (onglet "Modèles") les affiche automatiquement,
# en lecture seule (reference_only=true), aux côtés des 4 modèles classiques.
import json
import os

reference = {
    "model": "DistilBERT (fine-tuné)",
    "accuracy": round(bert_results["eval_accuracy"] * 100, 2),
    "precision": round(bert_results["eval_precision"] * 100, 2),
    "recall": round(bert_results["eval_recall"] * 100, 2),
    "f1_score": round(bert_results["eval_f1"] * 100, 2),
    "training_time_sec": round(bert_train_time, 2),
    "reference_only": True,
    "source": "notebooks/transformer_comparison.ipynb",
    "note": "Évalué séparément sur un sous-échantillon, non entraîné ni servi par l'application.",
}

output_path = os.path.join("..", "saved_models", "transformer_reference.json")
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(reference, f, ensure_ascii=False, indent=2)

print(f"Résultats écrits dans {output_path}")
reference

Résultats écrits dans ..\saved_models\transformer_reference.json


{'model': 'DistilBERT (fine-tuné)',
 'accuracy': 88.8,
 'precision': 90.63,
 'recall': 87.08,
 'f1_score': 88.82,
 'training_time_sec': 9790.75,
 'reference_only': True,
 'source': 'notebooks/transformer_comparison.ipynb',
 'note': "Évalué séparément sur un sous-échantillon, non entraîné ni servi par l'application."}